# Learning Module: First-Order Logic (FOL) RKA 25

## 1. Introduction

Propositional logic only knows whole **facts** (`P12` = "there is a pit in [1,2]"). As a result, the same rule must be written repeatedly for every square. Example: the breeze rule in the Wumpus World must be written for **16 squares**.

**First-Order Logic (FOL)** / predicate logic is more expressive because the world is viewed as consisting of:

| Component | Example |
|---|---|
| **Objects** (constants) | `John`, `Ani`, `M1` |
| **Relations** (predicates) | `Parent(Ani, Budi)`, `King(John)` |
| **Functions** | `Mother(Budi)` → another object |
| **Variables** | `x`, `y`, `z` |
| **Quantifiers** | ∀ (for all), ∃ (there exists) |

With FOL, the breeze rule only needs to be written once:

```
∀s  Breezy(s) ⇔ ∃r Adjacent(r, s) ∧ Pit(r)
```

In this module we implement from scratch:
1. Quantifier evaluation on finite models
2. **Substitution** and **Unification**
3. FOL **Forward Chaining**
4. FOL **Backward Chaining**

Reference: Russell & Norvig, *Artificial Intelligence: A Modern Approach*, Chapters 8 and 9.

## 2. Quantifiers and Models

### 2.1 Universal and Existential Quantifiers

| Quantifier | Notation | Meaning | Natural connective |
|---|---|---|---|
| Universal | `∀x P(x)` | P is true for **every** object | ⇒ (implication) |
| Existential | `∃x P(x)` | P is true for **at least one** object | ∧ (conjunction) |

Common mistakes:
- `∀x King(x) ∧ Person(x)` means "every object is a king **and** a person". This is wrong. The correct form: `∀x King(x) ⇒ Person(x)`.
- `∃x Crown(x) ⇒ OnHead(x, John)` is true as soon as there is any object that is not a crown. The correct form: `∃x Crown(x) ∧ OnHead(x, John)`.

De Morgan's laws for quantifiers:
```
¬∀x P(x)  ≡  ∃x ¬P(x)
¬∃x P(x)  ≡  ∀x ¬P(x)
```

### 2.2 Evaluating Quantifiers on a Finite Model

If the domain of objects is **finite**, `∀` can be computed with `all()` and `∃` with `any()` in Python.

In [1]:
# Model: domain of objects + predicate interpretation (as sets of tuples)
domain = ['Ani', 'Budi', 'Citra', 'Dodi', 'Eka']

Person = {'Ani', 'Budi', 'Citra', 'Dodi', 'Eka'}
Female = {'Ani', 'Citra', 'Eka'}
Parent = {('Ani', 'Budi'), ('Ani', 'Citra'), ('Budi', 'Dodi'), ('Citra', 'Eka')}
Loves  = {('Ani', 'Budi'), ('Budi', 'Ani'), ('Citra', 'Ani'), ('Dodi', 'Ani'), ('Eka', 'Ani')}

def forall(pred):  return all(pred(x) for x in domain)
def exists(pred):  return any(pred(x) for x in domain)
def implies(a, b): return (not a) or b

In [2]:
# ∀x Female(x) ⇒ Person(x)
print("∀x Female(x) ⇒ Person(x)        :", forall(lambda x: implies(x in Female, x in Person)))

# ∃x Parent(x, Citra)
print("∃x Parent(x, Citra)              :", exists(lambda x: (x, 'Citra') in Parent))

# ∀x ∃y Loves(x, y)   -> everyone loves someone
print("∀x ∃y Loves(x, y)                :", forall(lambda x: exists(lambda y: (x, y) in Loves)))

# ∃y ∀x Loves(x, y)   -> there is someone loved by everyone
print("∃y ∀x Loves(x, y)                :", exists(lambda y: forall(lambda x: (x, y) in Loves)))

# ∃y ∀x (x ≠ y ⇒ Loves(x, y))  -> there is someone loved by everyone else
print("∃y ∀x (x≠y ⇒ Loves(x, y))        :", exists(lambda y: forall(lambda x: implies(x != y, (x, y) in Loves))))

∀x Female(x) ⇒ Person(x)        : True
∃x Parent(x, Citra)              : True
∀x ∃y Loves(x, y)                : True
∃y ∀x Loves(x, y)                : False
∃y ∀x (x≠y ⇒ Loves(x, y))        : True


**Quantifier order matters!**
- `∀x ∃y Loves(x, y)` → everyone has someone they love (possibly different people).
- `∃y ∀x Loves(x, y)` → there is **one** person loved by **everyone** (including themselves). The result is False because Ani does not love herself.
- If "themselves" is excluded, Ani is loved by everyone else, so the result is True.

> This approach only works when the domain is finite and small. For general inference we use **unification** and **chaining**.

## 3. Representing FOL Sentences in Python

We use the AIMA convention:
- **Variable**: a string starting with a lowercase letter → `'x'`, `'y'`, `'z1'`
- **Constant**: a string starting with an uppercase letter → `'John'`, `'M1'`
- **Atom / function**: a tuple `(name, arg1, arg2, ...)` whose name starts with an uppercase letter

| FOL | Python |
|---|---|
| `King(John)` | `('King', 'John')` |
| `Knows(John, x)` | `('Knows', 'John', 'x')` |
| `Knows(y, Mother(y))` | `('Knows', 'y', ('Mother', 'y'))` |

In [3]:
def is_variable(x):
    return isinstance(x, str) and x[:1].islower()


def fmt(x):
    # Convert a tuple into readable FOL notation
    if isinstance(x, tuple):
        return f"{x[0]}({', '.join(fmt(a) for a in x[1:])})"
    return str(x)


def fmt_theta(theta):
    if theta is None:
        return "FAIL"
    # show the final value of each variable (substitution chains resolved)
    return "{" + ", ".join(f"{k}/{fmt(subst(theta, v))}" for k, v in theta.items()) + "}"


print(fmt(('Knows', 'y', ('Mother', 'y'))))
print(is_variable('x'), is_variable('John'))

Knows(y, Mother(y))
True False


## 4. Substitution and Unification

### 4.1 Substitution

A **substitution** θ is a mapping from variables to terms, written `{x/John, y/Mother(John)}`. `SUBST(θ, α)` replaces every variable in α according to θ.

In [4]:
def subst(theta, x):
    if is_variable(x):
        return subst(theta, theta[x]) if x in theta else x
    if isinstance(x, tuple):
        return tuple(subst(theta, a) for a in x)
    return x


print(fmt(subst({'x': 'John', 'y': 'Richard'}, ('Brother', 'x', 'y'))))


Brother(John, Richard)


### 4.2 Unification

**Unification** finds a substitution θ that makes two sentences **identical**:

```
UNIFY(p, q) = θ  such that  SUBST(θ, p) = SUBST(θ, q)
```

Algorithm (AIMA Fig. 9.1):
1. If `x == y` → already identical, return θ.
2. If `x` is a variable → `unify_var(x, y)`. Likewise if `y` is a variable.
3. If both are tuples of the same length → unify element by element.
4. Otherwise → **fail**.

**Occurs check**: a variable `x` must not be unified with a term that contains `x` itself, e.g. `x` with `Mother(x)`. Without this check an infinite term `Mother(Mother(Mother(...)))` would be formed.

In [5]:
def unify(x, y, theta=None):
    # Return a substitution (dict) that makes x and y identical, or None on failure
    if theta is None:
        theta = {}
    if x == y:
        return theta
    if is_variable(x):
        return unify_var(x, y, theta)
    if is_variable(y):
        return unify_var(y, x, theta)
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y):
        for xi, yi in zip(x, y):
            theta = unify(xi, yi, theta)
            if theta is None:
                return None
        return theta
    return None


def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    if is_variable(x) and x in theta:
        return unify(var, theta[x], theta)
    if occur_check(var, x, theta):
        return None
    return {**theta, var: x}


def occur_check(var, x, theta):
    if var == x:
        return True
    if is_variable(x) and x in theta:
        return occur_check(var, theta[x], theta)
    if isinstance(x, tuple):
        return any(occur_check(var, a, theta) for a in x)
    return False

### 4.3 Unification Examples (AIMA Section 9.2.2)

Query: `Knows(John, x)` ("whom does John know?")

In [6]:
query = ('Knows', 'John', 'x')
kb_sentences = [
    ('Knows', 'John', 'Jane'),
    ('Knows', 'y', 'Bill'),
    ('Knows', 'y', ('Mother', 'y')),
    ('Knows', 'x', 'Elizabeth'),
]

for s in kb_sentences:
    theta = unify(query, s)
    print(f"UNIFY({fmt(query)}, {fmt(s):22}) = {fmt_theta(theta)}")
    if theta is not None:
        print(f"    substitution result: {fmt(subst(theta, query))}")

UNIFY(Knows(John, x), Knows(John, Jane)     ) = {x/Jane}
    substitution result: Knows(John, Jane)
UNIFY(Knows(John, x), Knows(y, Bill)        ) = {y/John, x/Bill}
    substitution result: Knows(John, Bill)
UNIFY(Knows(John, x), Knows(y, Mother(y))   ) = {y/John, x/Mother(John)}
    substitution result: Knows(John, Mother(John))
UNIFY(Knows(John, x), Knows(x, Elizabeth)   ) = FAIL


The last case **fails** because the variable `x` is used in both sentences: `x` cannot be both `John` and `Elizabeth` at the same time.

Yet `∀x Knows(x, Elizabeth)` means *everyone* knows Elizabeth, including John. The solution is **standardizing apart**: rename the variables in one of the sentences so they do not clash.

In [7]:
import itertools

_counter = itertools.count()

def standardize_apart(x, mapping=None):
    # Replace every variable with a new unique name (x -> x_7, etc.)
    if mapping is None:
        mapping = {}
    if is_variable(x):
        if x not in mapping:
            mapping[x] = f"{x}_{next(_counter)}"
        return mapping[x]
    if isinstance(x, tuple):
        return tuple(standardize_apart(a, mapping) for a in x)
    return x


s = standardize_apart(('Knows', 'x', 'Elizabeth'))
print("After standardize apart:", fmt(s))
print("UNIFY =", fmt_theta(unify(query, s)))

After standardize apart: Knows(x_0, Elizabeth)
UNIFY = {x_0/John, x/Elizabeth}


In [8]:
# Occurs check: x cannot be unified with Mother(x)
print(fmt_theta(unify('x', ('Mother', 'x'))))

FAIL


## 5. Definite Clause Knowledge Base

As in propositional logic, efficient FOL inference uses **definite clauses**: a conjunction of positive atoms ⇒ a single positive atom. Variables are assumed to be universally quantified (∀).

We represent rules as `Rule(premises, conclusion)`. A fact is a rule with no premises.

### 5.1 Case Study: Colonel West's Crime (AIMA Section 9.3.1)

> *The law says that it is a crime for an American to sell weapons to hostile nations. The country Nono, an enemy of America, has some missiles, and all of its missiles were sold to it by Colonel West, who is American.*

| FOL sentence | Description |
|---|---|
| `American(x) ∧ Weapon(y) ∧ Sells(x, y, z) ∧ Hostile(z) ⇒ Criminal(x)` | criminal rule |
| `Owns(Nono, M1)`, `Missile(M1)` | Nono owns missile M1 (from ∃x Owns(Nono,x) ∧ Missile(x)) |
| `Missile(x) ∧ Owns(Nono, x) ⇒ Sells(West, x, Nono)` | all of Nono's missiles were sold by West |
| `Missile(x) ⇒ Weapon(x)` | missiles are weapons |
| `Enemy(x, America) ⇒ Hostile(x)` | an enemy of America is *hostile* |
| `American(West)` | West is American |
| `Enemy(Nono, America)` | Nono is an enemy of America |

In [9]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Rule:
    premises: tuple
    conclusion: tuple

    def __repr__(self):
        if not self.premises:
            return fmt(self.conclusion)
        return " ∧ ".join(fmt(p) for p in self.premises) + " ⇒ " + fmt(self.conclusion)


def fact(atom):
    return Rule((), atom)


crime_kb = [
    Rule((('American', 'x'), ('Weapon', 'y'), ('Sells', 'x', 'y', 'z'), ('Hostile', 'z')),
         ('Criminal', 'x')),
    fact(('Owns', 'Nono', 'M1')),
    fact(('Missile', 'M1')),
    Rule((('Missile', 'x'), ('Owns', 'Nono', 'x')), ('Sells', 'West', 'x', 'Nono')),
    Rule((('Missile', 'x'),), ('Weapon', 'x')),
    Rule((('Enemy', 'x', 'America'),), ('Hostile', 'x')),
    fact(('American', 'West')),
    fact(('Enemy', 'Nono', 'America')),
]

for r in crime_kb:
    print(r)


American(x) ∧ Weapon(y) ∧ Sells(x, y, z) ∧ Hostile(z) ⇒ Criminal(x)
Owns(Nono, M1)
Missile(M1)
Missile(x) ∧ Owns(Nono, x) ⇒ Sells(West, x, Nono)
Missile(x) ⇒ Weapon(x)
Enemy(x, America) ⇒ Hostile(x)
American(West)
Enemy(Nono, America)


## 6. FOL Forward Chaining

### 6.1 Algorithm

Forward chaining is **data-driven**: start from the facts, fire rules whose premises are satisfied, add the conclusions as new facts, and repeat until no new facts appear (*fixed point*).

For each rule, we find every substitution θ that makes **all premises** match existing facts (*pattern matching*), then add `SUBST(θ, conclusion)`.

Since all facts are **ground** (contain no variables), we do not need to standardize apart in forward chaining.

In [10]:
def match_premises(premises, facts, theta):
    # Generator: every θ that makes each premise match one of the facts
    if not premises:
        yield theta
        return
    first, rest = premises[0], premises[1:]
    for f in facts:
        t = unify(first, f, theta)
        if t is not None:
            yield from match_premises(rest, facts, t)


def fol_fc_ask(kb, query, verbose=True):
    facts = {r.conclusion for r in kb if not r.premises}
    rules = [r for r in kb if r.premises]
    iteration = 0
    while True:
        iteration += 1
        new = set()
        for rule in rules:
            for theta in match_premises(rule.premises, facts, {}):
                inferred = subst(theta, rule.conclusion)
                if inferred not in facts and inferred not in new:
                    new.add(inferred)
                    if verbose:
                        print(f"[iteration {iteration}] {fmt(inferred):28} from  {rule}")
        if not new:
            break
        facts |= new
    # Answers: every substitution that makes the query match a fact
    answers = [unify(query, f) for f in facts]
    return [a for a in answers if a is not None]

In [11]:
answers = fol_fc_ask(crime_kb, ('Criminal', 'x'))
print("\nAnswers for Criminal(x):", [fmt_theta(a) for a in answers])

[iteration 1] Sells(West, M1, Nono)        from  Missile(x) ∧ Owns(Nono, x) ⇒ Sells(West, x, Nono)
[iteration 1] Weapon(M1)                   from  Missile(x) ⇒ Weapon(x)
[iteration 1] Hostile(Nono)                from  Enemy(x, America) ⇒ Hostile(x)
[iteration 2] Criminal(West)               from  American(x) ∧ Weapon(y) ∧ Sells(x, y, z) ∧ Hostile(z) ⇒ Criminal(x)

Answers for Criminal(x): ['{x/West}']


The result above matches the forward chaining *proof tree* in AIMA Fig. 9.4:
- **Iteration 1**: `Sells(West, M1, Nono)`, `Weapon(M1)`, `Hostile(Nono)`
- **Iteration 2**: `Criminal(West)`
- **Iteration 3**: no new facts → *fixed point*

## 7. FOL Backward Chaining

### 7.1 Algorithm

Backward chaining is **goal-driven**: start from the **query**, find rules whose conclusion can be unified with the query, then prove their premises recursively (DFS).

The algorithm (AIMA Fig. 9.6) consists of two mutually recursive functions:
- **`bc_or(goal)`**: the goal can be proved by any rule (**OR**).
- **`bc_and(goals)`**: all sub-goals must be proved (**AND**).

Unlike forward chaining, rules here **must be standardized apart** every time they are used, because variables from the goal and the rule may clash.

In [12]:
def bc_or(kb, goal, theta, depth, verbose):
    for rule in kb:
        mapping = {}
        conclusion = standardize_apart(rule.conclusion, mapping)
        premises = tuple(standardize_apart(p, mapping) for p in rule.premises)
        t = unify(goal, conclusion, theta)
        if t is not None:
            if verbose:
                print("  " * depth + f"goal {fmt(subst(theta, goal))}  ←  {rule}")
            yield from bc_and(kb, premises, t, depth + 1, verbose)


def bc_and(kb, goals, theta, depth, verbose):
    if not goals:
        yield theta
        return
    first, rest = goals[0], goals[1:]
    for t1 in bc_or(kb, subst(theta, first), theta, depth, verbose):
        yield from bc_and(kb, rest, t1, depth, verbose)


def fol_bc_ask(kb, query, verbose=False):
    # Return every unique answer (the query with the substitution applied)
    results = []
    for theta in bc_or(kb, query, {}, 0, verbose):
        ans = subst(theta, query)
        if ans not in results:
            results.append(ans)
    return results

In [13]:
answers = fol_bc_ask(crime_kb, ('Criminal', 'x'), verbose=True)
print("\nAnswers:", [fmt(a) for a in answers])

goal Criminal(x)  ←  American(x) ∧ Weapon(y) ∧ Sells(x, y, z) ∧ Hostile(z) ⇒ Criminal(x)
  goal American(x_1)  ←  American(West)
  goal Weapon(y_2)  ←  Missile(x) ⇒ Weapon(x)
    goal Missile(x_14)  ←  Missile(M1)
  goal Sells(West, M1, z_3)  ←  Missile(x) ∧ Owns(Nono, x) ⇒ Sells(West, x, Nono)
    goal Missile(M1)  ←  Missile(M1)
    goal Owns(Nono, M1)  ←  Owns(Nono, M1)
  goal Hostile(Nono)  ←  Enemy(x, America) ⇒ Hostile(x)
    goal Enemy(Nono, America)  ←  Enemy(Nono, America)

Answers: ['Criminal(West)']


The trace above shows the *backward* proof tree (AIMA Fig. 9.7): to prove `Criminal(x)`, the agent proves `American(x)` → `x = West`, then `Weapon(y)`, `Sells(West, y, z)`, and `Hostile(z)` in order.

### 7.2 Case Study: Family Tree

Backward chaining can also answer queries with many answers, including **recursive** rules such as `Ancestor`.

```
       Ani
      /    \
   Budi    Citra
    |        |
   Dodi     Eka
```

In [14]:
family_kb = [
    fact(('Parent', 'Ani', 'Budi')),
    fact(('Parent', 'Ani', 'Citra')),
    fact(('Parent', 'Budi', 'Dodi')),
    fact(('Parent', 'Citra', 'Eka')),
    # Ancestor(x, y) ⇐ Parent(x, y)
    Rule((('Parent', 'x', 'y'),), ('Ancestor', 'x', 'y')),
    # Ancestor(x, z) ⇐ Parent(x, y) ∧ Ancestor(y, z)
    Rule((('Parent', 'x', 'y'), ('Ancestor', 'y', 'z')), ('Ancestor', 'x', 'z')),
]

print("Descendants of Ani:", [fmt(a) for a in fol_bc_ask(family_kb, ('Ancestor', 'Ani', 'w'))])
print("Ancestors of Eka  :", [fmt(a) for a in fol_bc_ask(family_kb, ('Ancestor', 'w', 'Eka'))])
print("Ancestor(Budi,Eka):", fol_bc_ask(family_kb, ('Ancestor', 'Budi', 'Eka')))

Descendants of Ani: ['Ancestor(Ani, Budi)', 'Ancestor(Ani, Citra)', 'Ancestor(Ani, Dodi)', 'Ancestor(Ani, Eka)']
Ancestors of Eka  : ['Ancestor(Citra, Eka)', 'Ancestor(Ani, Eka)']
Ancestor(Budi,Eka): []


An empty list `[]` means the query **cannot be proved** from the KB.

> ⚠️ DFS-based backward chaining can **get stuck in an infinite loop** on *left-recursive* rules, e.g. `Ancestor(x, z) ⇐ Ancestor(x, y) ∧ Parent(y, z)`. Therefore, put the base predicate (`Parent`) before the recursive predicate.

## 8. Comparison and Summary

| Aspect | Forward Chaining | Backward Chaining |
|---|---|---|
| Direction | Data-driven (facts → conclusions) | Goal-driven (query → facts) |
| Strategy | Iterate until *fixed point* | Recursive DFS |
| Standardize apart | Not needed (ground facts) | Required every time a rule is used |
| Strength | Derives **all** consequences of the KB | Only works on what is relevant to the query |
| Weakness | May derive many irrelevant facts | May loop & repeat work |
| Example applications | Production systems, *deductive databases* | Prolog, expert systems |

| Propositional Logic | First-Order Logic |
|---|---|
| Whole facts (`P12`) | Objects, relations, functions |
| No variables | Variables + quantifiers ∀, ∃ |
| Identical symbol matching | **Unification** |
| Entailment is *decidable* | Entailment is *semi-decidable* |

## 9. Exercises (Homework)
<font color="blue">Exercise 7</font>

### Homework 1 — Unification

Determine the results of the following unifications **by hand** first, then check them with the `unify` function. Explain why if a unification fails.

| No | p | q |
|---|---|---|
| a | `Older(Father(y), y)` | `Older(Father(x), John)` |
| b | `Knows(Father(y), y)` | `Knows(x, x)` |
| c | `P(A, y, F(y))` | `P(x, B, z)` |
| d | `Q(x, G(x))` | `Q(G(y), y)` |

In [15]:
pairs = [
    (('Older', ('Father', 'y'), 'y'), ('Older', ('Father', 'x'), 'John')),
    (_____, _____),   # b
    (_____, _____),   # c
    (_____, _____),   # d
]

for p, q in pairs:
    print(f"UNIFY({fmt(p)}, {fmt(q)}) = {fmt_theta(unify(p, q))}")


NameError: name '_____' is not defined

**Analysis (write here):**
- a: ...
- b: ...
- c: ...
- d: ...

### Homework 2 — Family Tree

Given the following family tree:

```
          Hasan (M) ── Siti (F)
          /                  \
     Rudi (M) ── Maya (F)    Lina (F) ── Joko (M)
       /     \                   |
  Tono (M)  Wati (F)          Sari (F)
```

**Task:** Complete the following rules as definite clauses:
1. `Mother(x, y)`: x is the mother of y
2. `Grandparent(x, z)`: x is a grandparent of z
3. `Grandmother(x, z)`: x is the grandmother of z
4. `Ancestor(x, z)`: x is an ancestor of z (recursive)

Then answer the queries using **forward chaining** and **backward chaining**:
- Who are Hasan's grandchildren?
- Who is Sari's grandmother?
- Who are Siti's descendants?

In [16]:
pr_kb = [
    fact(('Parent', 'Hasan', 'Rudi')), fact(('Parent', 'Siti', 'Rudi')),
    fact(('Parent', 'Hasan', 'Lina')), fact(('Parent', 'Siti', 'Lina')),
    fact(('Parent', 'Rudi', 'Tono')),  fact(('Parent', 'Maya', 'Tono')),
    fact(('Parent', 'Rudi', 'Wati')),  fact(('Parent', 'Maya', 'Wati')),
    fact(('Parent', 'Lina', 'Sari')),  fact(('Parent', 'Joko', 'Sari')),
    fact(('Female', 'Siti')), fact(('Female', 'Maya')), fact(('Female', 'Lina')),
    fact(('Female', 'Wati')), fact(('Female', 'Sari')),

    # 1. Mother(x, y) ⇐ Parent(x, y) ∧ Female(x)
    Rule((('Parent', 'x', 'y'), ('Female', 'x')), ('Mother', 'x', 'y')),

    # 2. Grandparent(x, z) ⇐ ...
    Rule(_____, ('Grandparent', 'x', 'z')),

    # 3. Grandmother(x, z) ⇐ ...
    Rule(_____, ('Grandmother', 'x', 'z')),

    # 4. Ancestor(x, z) ⇐ ...   (two rules: base case and recursive case)
    Rule(_____, ('Ancestor', 'x', 'z')),
    Rule(_____, ('Ancestor', 'x', 'z')),
]

NameError: name '_____' is not defined

In [17]:
# Forward chaining
print("Hasan's grandchildren (FC):", [fmt_theta(t) for t in fol_fc_ask(pr_kb, ('Grandparent', 'Hasan', 'w'), verbose=False)])

# Backward chaining
print("Hasan's grandchildren     :", [fmt(a) for a in fol_bc_ask(pr_kb, ('Grandparent', 'Hasan', 'w'))])
print("Sari's grandmother        :", [fmt(a) for a in fol_bc_ask(pr_kb, ('Grandmother', 'w', 'Sari'))])
print("Siti's descendants        :", [fmt(a) for a in fol_bc_ask(pr_kb, ('Ancestor', 'Siti', 'w'))])

NameError: name 'pr_kb' is not defined

**Analysis questions:**
1. Why does backward chaining need *standardize apart*, while forward chaining does not?
2. What happens if the recursive `Ancestor` rule is written as `Ancestor(x, z) ⇐ Ancestor(x, y) ∧ Parent(y, z)`? Try it and explain.

**Answer:** ...